# PhoBERT Fine-tuning — Colab Version
Training sentiment analysis với data đã augment. Chạy các cell theo thứ tự từ trên xuống.

In [ ]:
# ── Bước 1: Mount Google Drive ───────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os, sys

# !! Sửa path này thành đúng folder project trên Drive của bạn
PROJECT_ROOT = '/content/drive/MyDrive/project_dat391'

os.chdir(PROJECT_ROOT)
sys.path.append(PROJECT_ROOT)

print('Working dir:', os.getcwd())
print('Files:', os.listdir('.'))

In [ ]:
# ── Bước 2: Cài thư viện (chỉ cần chạy 1 lần) ───────────────────
!pip install transformers datasets torch scikit-learn -q

In [ ]:
# ── Bước 3: Kiểm tra GPU ─────────────────────────────────────────
import torch

if torch.cuda.is_available():
    print(f'✓ GPU: {torch.cuda.get_device_name(0)}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('✗ Không có GPU — vào Runtime → Change runtime type → T4 GPU')

In [ ]:
# ── Bước 4: Load data đã processed ──────────────────────────────
from src.phobert_pipeline import (
    load_splits,
    build_hf_datasets,
    tokenize_datasets,
    build_trainer,
    evaluate_on_test,
    save_metrics,
)

train_df, val_df, test_df = load_splits(
    'data/processed/train_split.csv',
    'data/processed/val_split.csv',
    'data/processed/test_split_from_train.csv',
)

print(f'Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}')
print('Label distribution (train):')
print(train_df['label'].value_counts())

In [ ]:
# ── Bước 5: Build HuggingFace datasets + tokenize ───────────────
train_ds, val_ds, test_ds = build_hf_datasets(train_df, val_df, test_df)
tokenizer, train_tok, val_tok, test_tok = tokenize_datasets(train_ds, val_ds, test_ds)

print('Tokenization done!')
print('Sample:', train_tok[0])

In [ ]:
# ── Bước 6: Định nghĩa WeightedTrainer ──────────────────────────
# Dùng class_weights thay cho undersample — giữ nguyên toàn bộ data
from transformers import Trainer
from torch.nn import CrossEntropyLoss

class WeightedTrainer(Trainer):
    def __init__(self, class_weights, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits  = outputs.logits
        weights = self.class_weights.to(logits.device)
        loss = CrossEntropyLoss(weight=weights)(logits, labels)
        return (loss, outputs) if return_outputs else loss

# Load class_weights đã tính sẵn
class_weights = torch.load(
    'data/processed/class_weights.pt',
    map_location='cpu',
    weights_only=True
)
print(f'Class weights — neg: {class_weights[0]:.4f} | pos: {class_weights[1]:.4f}')

In [ ]:
# ── Bước 7: Build trainer & bắt đầu training ────────────────────
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
)
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

MODEL_NAME   = 'vinai/phobert-base'
OUTPUT_DIR   = 'models/phobert_binary_best'
LOGGING_DIR  = 'models/logs'

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy':  accuracy_score(labels, preds),
        'f1_macro':  f1_score(labels, preds, average='macro'),
        'precision': precision_score(labels, preds, average='macro', zero_division=0),
        'recall':    recall_score(labels, preds, average='macro', zero_division=0),
    }

training_args = TrainingArguments(
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = 3,
    per_device_train_batch_size = 32,
    per_device_eval_batch_size  = 64,
    learning_rate               = 2e-5,
    warmup_ratio                = 0.1,
    weight_decay                = 0.01,
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'f1_macro',
    greater_is_better           = True,
    logging_dir                 = LOGGING_DIR,
    logging_steps               = 100,
    fp16                        = torch.cuda.is_available(),  # tự bật FP16 nếu có GPU
    report_to                   = 'none',
)

trainer = WeightedTrainer(
    class_weights   = class_weights,
    model           = model,
    args            = training_args,
    train_dataset   = train_tok,
    eval_dataset    = val_tok,
    compute_metrics = compute_metrics,
)

print('Bắt đầu training...')
trainer.train()

In [ ]:
# ── Bước 8: Evaluate trên test set ──────────────────────────────
print('=== Kết quả trên Test set ===')
test_results = trainer.evaluate(eval_dataset=test_tok)
for k, v in test_results.items():
    if isinstance(v, float):
        print(f'  {k}: {v:.4f}')

In [ ]:
# ── Bước 9: Lưu model best ───────────────────────────────────────
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'✓ Model saved → {OUTPUT_DIR}')
print('Files:', os.listdir(OUTPUT_DIR))

In [ ]:
# ── Bước 10: Test nhanh predict ─────────────────────────────────
from transformers import pipeline

classifier = pipeline(
    'text-classification',
    model=OUTPUT_DIR,
    tokenizer=OUTPUT_DIR,
    device=0 if torch.cuda.is_available() else -1,
)

test_sentences = [
    'Món ăn ngon tuyệt vời, phục vụ nhiệt tình!',
    'Dịch vụ tệ quá, chờ cả tiếng không có ai hỏi thăm.',
    'Phim này hay, diễn xuất tốt và cốt truyện hấp dẫn.',
    'Sản phẩm kém chất lượng, không như mô tả.',
]

for sent in test_sentences:
    result = classifier(sent)[0]
    label  = 'POSITIVE' if result['label'] == 'LABEL_1' else 'NEGATIVE'
    print(f'[{label} {result["score"]:.2f}] {sent}')